# Validate packed results

Use this notebook for an immediate sanity check on one compact `.h5` session before running the full webpage report.

## 1. Setup

Run from `passive_interval_oddball_202412/notebooks`. Change `H5_PATH` to any session under `results_pack/<subject>/<session>/<session>.h5`.

In [ ]:
from pathlib import Path
import os

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)

H5_PATH = repo / 'results_pack' / 'YH21SC' / 'V1_random_01' / 'V1_random_01.h5'
print(H5_PATH)
print('exists:', H5_PATH.exists())

## 2. Inspect the file contents

This confirms that the released h5 contains masks, labels, neural traces, stimulus timing, voltage traces, and pupil data.

In [ ]:
import h5py

with h5py.File(H5_PATH, 'r') as f:
    def show(name, obj):
        if hasattr(obj, 'shape'):
            print(f'{name:35s} shape={str(obj.shape):18s} dtype={obj.dtype}')
        else:
            print(name)
    f.visititems(show)

## 3. Load small validation arrays

The full `dff` matrix can be large, but one packed session should still be easy to inspect locally.

In [ ]:
import numpy as np

with h5py.File(H5_PATH, 'r') as f:
    labels = f['labels'][:]
    dff = f['neural_trials']['dff'][:]
    time = f['neural_trials']['time'][:]
    stim_labels = f['neural_trials']['stim_labels'][:]
    camera_pupil = f['neural_trials']['camera_pupil'][:]

print('labels:', labels.shape, dict(zip(*np.unique(labels, return_counts=True))))
print('dff:', dff.shape, 'finite fraction:', np.isfinite(dff).mean().round(4))
print('time:', time.shape, 'range:', (float(time[0]), float(time[-1])))
print('stim_labels:', stim_labels.shape)
print('camera_pupil:', camera_pupil.shape, 'finite fraction:', np.isfinite(camera_pupil).mean().round(4))

## 4. Basic integrity checks

These are intentionally simple checks for common data-loading mistakes: mismatched frame counts, non-monotonic time, missing stimulus labels, or all-NaN traces.

In [ ]:
checks = {
    'dff has neuron x frame shape': dff.ndim == 2,
    'time matches dff frames': len(time) == dff.shape[1],
    'time is increasing': np.all(np.diff(time) > 0),
    'has stimulus labels': stim_labels.ndim == 2 and stim_labels.shape[0] > 0,
    'dff has finite values': np.isfinite(dff).any(),
}

for name, passed in checks.items():
    print(('OK   ' if passed else 'FAIL ') + name)

if not all(checks.values()):
    raise ValueError('One or more validation checks failed.')

## 5. Quick visual validation

The first plot shows the session-average fluorescence over the first frames. The second plot aligns a small population average around visual stimulus onsets from `stim_labels[:, 0]`.

In [ ]:
import matplotlib.pyplot as plt

n_frames = min(5000, dff.shape[1])
mean_trace = np.nanmean(dff[:, :n_frames], axis=0)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(time[:n_frames], mean_trace, lw=1)
ax.set_title('Population mean dF/F, first frames')
ax.set_xlabel('time')
ax.set_ylabel('mean dF/F')
plt.show()

In [ ]:
left_frames = 20
right_frames = 80
event_times = stim_labels[:, 0]
event_idx = np.searchsorted(time, event_times)
event_idx = event_idx[(event_idx > left_frames) & (event_idx + right_frames < dff.shape[1])]
event_idx = event_idx[:200]

if len(event_idx) == 0:
    raise ValueError('No stimulus events fall inside the available frame window.')

pop = np.nanmean(dff, axis=0)
aligned = np.stack([pop[i-left_frames:i+right_frames] for i in event_idx])
x = np.arange(-left_frames, right_frames)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, np.nanmean(aligned, axis=0), lw=2)
ax.axvline(0, color='k', lw=1, ls='--')
ax.set_title(f'Mean response around {len(event_idx)} stimulus onsets')
ax.set_xlabel('frames from stimulus onset')
ax.set_ylabel('population mean dF/F')
plt.show()

## 6. Next step

If the checks pass and the plots look reasonable, run the packed report from the project root:

```bash
python main.py --config_list YH21SC --use_packed
```

The report is saved to `results/YH21SC_V1_passive.html`.